# Runs all QM7 experiments with the full dataset!

(Remember that the atomization energy is the energy required to form 1 mol of gaseous atoms from 1 mol of the molecule in its standard state under standard conditions)

Apparently 0.015 is what multitask will get, and 0.0143 is the best (MPNN)

In [ ]:
import sys
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

import deepchem as dc

import tensorflow as tf
import os
import sys
import rdkit
import h5py
import helper_functions as h

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri
import rdkit.Chem as Chem
import rdkit.Chem.AllChem as AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
import mpl_toolkits.mplot3d
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from collections import Counter

print("TensorFlow version: " + tf.__version__)

from notebook.services.config import ConfigManager
ConfigManager().update('notebook', {'ExecuteTime': {
   	'display_absolute_timestamps': False,
    'relative_timing_update_period': 5,
    'template': {
    	'executed': 'started ${start_time}, finished in ${duration}',
    }
}})

# topology stuff
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.plotting import plot_diagram
from gtda.diagrams import PersistenceEntropy
from gtda.diagrams import NumberOfPoints
from gtda.diagrams import Amplitude
from sklearn.pipeline import make_union, Pipeline

# fixc this at some point
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

import projection
from projection.molecule import Molecule
from projection.pdbmolecule import PDBMolecule
from projection.mol2molecule import Mol2Molecule

import helper_functions as h
#from projection.face import Face

# $UN THIS
data_dir=r'F:\Nextcloud\science\Datasets\topol_datasets'
results_dir=r"F:\Nextcloud\science\results\topology_and_graphs\QM7"
test_file='qm7.csv'
data_file_name='qm7_topological_features.hdf5'
make_dataset=False # whether to recalc the dataset


print(f"DeepChem version: {dc.__version__}")

############################### settings for all experiments #################

num_repeats=3
num_epochs = 3#500

metric_labels=['mean_squared_error','pearson_r2_score',
               'mae_score', 'rmse']


metric1 = dc.metrics.Metric(dc.metrics.mean_squared_error)
metric2 = dc.metrics.Metric(dc.metrics.pearson_r2_score)
metric3 = dc.metrics.Metric(dc.metrics.mae_score)
metrics = [metric1, metric2, metric3]
selected_metric = 2 #which metric to use for callback

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.kernel_ridge import KernelRidge

In [ ]:
hdf5_file_name='qm7_topological_features.hdf5'
fh = h5py.File(os.path.join(data_dir,hdf5_file_name), 'r+')
num_of_rows, num_of_molecules = h.basic_info_hdf5_dataset(fh, label='molID')

In [ ]:
fh.keys()

In [ ]:
fh['u0_atom'][:]

In [ ]:
feature_name_list = ['pers_S_1', 'pers_S_2', 'pers_S_3',
                    'no_p_1', 'no_p_2', 'no_p_3',
                    'bottle_1', 'bottle_2', 'bottle_3',
                    'wasser_1', 'wasser_2', 'wasser_3',
                    'landsc_1', 'landsc_2', 'landsc_3',
                    'pers_img_1', 'pers_img_2', 'pers_img_3']

PCA_list = ['PCA_1', 'PCA_2', 'PCA_3',
           'PCA_4', 'PCA_5', 'PCA_6',
           'PCA_7', 'PCA_8', 'PCA_9',
           'PCA_10', 'PCA_11', 'PCA_12',
           'PCA_13', 'PCA_14', 'PCA_15',
           'PCA_16', 'PCA_17', 'PCA_18']

task_list = ['u0_atom']

## Functions

In [ ]:
metric_labels

In [ ]:
def load_all_hdf5(fh,
                  num_of_rows, 
                  column_headers):
    """If dataset is small enough, load it all into memory
    fh = file handle
    num_of_rows = number of rows of data (i.e. molecules)
    column_headers into the hdf5 file"""
    data = np.zeros((num_of_rows,len(column_headers)))
    for key_num in range(len(column_headers)):
        key = column_headers[key_num]
        #print(key_num)
        d=fh[key]
        data[:,key_num] = d
    return data

def do_transform(transformers, dataset):
    for transformer in transformers:
        dataset = transformer.transform(dataset)
    return dataset

################## not copied into helper yet ####################

def get_them_metrics(
    model,
    datasets,
    metrics,
    metric_labels,
    transformers=[],
):
    """gross function meh gets data tho"""
    ugh=[]
    for dataset in datasets:
        if transformers == []:
            egg=model.evaluate(
                dataset, 
                metrics)
        else:
            egg=model.evaluate(
                dataset, 
                metrics,
                transformers = transformers)
        for metric_label in metric_labels:
            if metric_label == 'rmse':
                ugh.append(np.sqrt(egg['mean_squared_error']))
            else:
                ugh.append(egg[metric_label])
    return ugh

def topol_regression_experiment(
    dataset,
    transformers,
    Splitter_Object,
    tasks,
    metrics,
    metric_selector,
    num_repeats=5,
    num_epochs=250,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse']):
    """runs repeated training with topol input features
    uses default multitask regressor class
    does splitting and transforming
    returns metrics
    
    dataset: overall original dataset, before splitting
    transformers: deepchem transformer
    Splitter_object: deepchem splitter
    num_epochs: num epochs if not early stopping
    metric_selector: whihc one to train wiht
    split fraction: train, val, test split as decimal
    patience: for early stopping
    metric_labels: add rmse here if you want it, do not add to metrics
    metrics: list of metrics"""

## in function

    out=[]

    frac_train=split_fraction[0]
    frac_valid=split_fraction[1]
    frac_test=split_fraction[2]
    train_scores, validate_scores, test_scores = [],[],[]
    print(f'Metric selected is {metric_labels[metric_selector]}')
    for i in range(num_repeats):
        # make datasets
        train_dataset, valid_dataset, test_dataset = Splitter_Object.train_valid_test_split(
        dataset=dataset,
        frac_train=frac_train,
        frac_valid=frac_valid,
        frac_test=frac_test)
        # transforms datasets wooo 
        train_dataset=do_transform(transformers, train_dataset)
        valid_dataset=do_transform(transformers, valid_dataset)
        test_dataset=do_transform(transformers, test_dataset)
        # for later
        datasets = [train_dataset, valid_dataset, test_dataset]
        # actual model here
        model = dc.models.MultitaskRegressor(
            n_tasks=len(tasks),
            n_features=len(train_dataset.X[3]),
            #layer_sizes=[1000,1000,500,20],
            #dropouts=0.2,
            #learning_rate=0.001,
            residual=True)
        callback = dc.models.ValidationCallback(
            valid_dataset, 
            patience, 
            metrics[metric_selector])
        # fit da model
        model.fit(train_dataset, nb_epoch=num_epochs, callbacks=callback)

        # little function to calc metrics on this data
        out.append(get_them_metrics(
            model,
            datasets,
            metrics,
            metric_labels,
            transformers))
    
    pd_out =pd.DataFrame(out, columns=['tr_mse','tr_r2','tr_mae', 'tr_rmse',
                              'val_mse', 'val_r2', 'val_mae','val_rmse',
                              'te_mse', 'te_r2', 'te_mae','te_rmse'])
    return pd_out

def no_transform_topol_regression_experiment(
    dataset,
    Splitter_Object,
    tasks,
    metrics,
    metric_selector,
    num_repeats=5,
    num_epochs=250,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse']):
    """runs repeated training with topol input features
    uses default multitask regressor class
    does splitting and transforming
    returns metrics
    
    dataset: overall original dataset, before splitting
    transformers: deepchem transformer
    Splitter_object: deepchem splitter
    num_epochs: num epochs if not early stopping
    metric_selector: whihc one to train wiht
    split fraction: train, val, test split as decimal
    patience: for early stopping
    metric_labels: add rmse here if you want it, do not add to metrics
    metrics: list of metrics"""

## in function

    out=[]

    frac_train=split_fraction[0]
    frac_valid=split_fraction[1]
    frac_test=split_fraction[2]
    train_scores, validate_scores, test_scores = [],[],[]
    print(f'Metric selected is {metric_labels[metric_selector]}')
    for i in range(num_repeats):
        # make datasets
        train_dataset, valid_dataset, test_dataset = Splitter_Object.train_valid_test_split(
        dataset=dataset,
        frac_train=frac_train,
        frac_valid=frac_valid,
        frac_test=frac_test)
        # transforms datasets wooo 
        #train_dataset=do_transform(transformers, train_dataset)
        #valid_dataset=do_transform(transformers, valid_dataset)
        #test_dataset=do_transform(transformers, test_dataset)
        # for later
        datasets = [train_dataset, valid_dataset, test_dataset]
        # actual model here
        model = dc.models.MultitaskRegressor(
            n_tasks=len(tasks),
            n_features=len(train_dataset.X[3]),
            #layer_sizes=[1000,1000,500,20],
            #dropouts=0.2,
            #learning_rate=0.001,
            residual=True)
        callback = dc.models.ValidationCallback(
            valid_dataset, 
            patience, 
            metrics[metric_selector])
        # fit da model
        model.fit(train_dataset, nb_epoch=num_epochs, callbacks=callback)

        # little function to calc metrics on this data
        out.append(get_them_metrics(
            model,
            datasets,
            metrics,
            metric_labels,
            transformers=[]))
    
    pd_out =pd.DataFrame(out, columns=['tr_mse','tr_r2','tr_mae', 'tr_rmse',
                              'val_mse', 'val_r2', 'val_mae','val_rmse',
                              'te_mse', 'te_r2', 'te_mae','te_rmse'])
    return pd_out



# Load data


## Create topological datasets

We use Numpy datasets to create the two toplogical datasets and two transformers (used as the controls use them and they are supposed to improve training). The topological dataset is normalised in X and y, PCA is only normalised in y as PCA is a normalisation.

This sorts out the datasets, transformers and splitters for the task

In [ ]:
fh['SMILES']

In [ ]:
## loading data from the hdf5 file
X_data = load_all_hdf5(
    fh,
    num_of_rows, 
    column_headers=feature_name_list)

PCA_X_data = load_all_hdf5(
    fh,
    num_of_rows, 
    column_headers=PCA_list)

y_data = load_all_hdf5(
    fh,
    num_of_rows, 
    column_headers=task_list)

SMILES_list = np.array(fh['SMILES'])

# making the datasets
topol_dataset = dc.data.DiskDataset.from_numpy(
    X_data, 
    y_data, 
    ids=SMILES_list)

pca_dataset = dc.data.DiskDataset.from_numpy(
    PCA_X_data, 
    y_data, 
    ids=SMILES_list)

# setting up the splitters for the task
Splitter_Object_tf = dc.splits.SingletaskStratifiedSplitter()
Splitter_Object_pca = dc.splits.SingletaskStratifiedSplitter()

# doing a transform on the data to make it easier for hte NN
# both y and x
transformers_tf = [
    dc.trans.NormalizationTransformer(
        transform_X=True, 
        dataset=topol_dataset),
    dc.trans.NormalizationTransformer(
        transform_y=True, 
        dataset=topol_dataset)]
# only y
transformers_pca = [
    dc.trans.NormalizationTransformer(
        transform_y=True, 
        dataset=pca_dataset)]



In [ ]:
plt.hist(y_data)
plt.xlabel('Atomisation energy')
plt.ylabel("No.")

# Test stuff

In [ ]:
# this line loads the  dataset - we are using the extended connectivity fingerprints here, 
# there are several featurizers for you to try
# the splitter splits the dataset for you, refer to the MoleculeNet paper to see which one you need

featurizer = dc.feat.WeaveFeaturizer()
tasks, datasets, transformers = dc.molnet.load_qm7(
    featurizer = 'Weave',
    splitter='random')

# the datasets object is already split into the train, validation and test dataset 
train_dataset, valid_dataset, test_dataset = datasets

# This transforms the dataset to try to deal with the unbalanced classes
transformer = dc.trans.BalancingTransformer(dataset=train_dataset)
train_dataset = transformer.transform(train_dataset)

transformer = dc.trans.BalancingTransformer(dataset=valid_dataset)
valid_dataset = transformer.transform(valid_dataset)

transformer = dc.trans.BalancingTransformer(dataset=test_dataset)
test_dataset = transformer.transform(test_dataset)

# WeaveModel

model = dc.models.WeaveModel(
    n_tasks = len(test_dataset.tasks), # size of y, we have one output task here: finding toxicity
    n_weave = 2,
    fully_connected_layer_sizes = [2000, 100],
    #n_atom_feat = 75,
    #n_pair_feat = 14,
    #n_hidden = 50,
    #n_graph_feat = 128,
    #conv_weight_init_stddevs = 0.03,
    #weight_init_stddevs = 0.01,
    #bias_init_consts = 0.0,
    #weight_decay_penalty = 0.0,
    #weight_decay_penalty_type = "l2",
    #dropouts = 0.2,
    #n_classes = 1,
    batch_size = 100,
    
    mode = "regression"
    )

############################################
# Now we fit the training dataset!         #
############################################
model.fit(test_dataset, nb_epoch=2)

In [ ]:
model.predict(test_dataset)[:5]

## DAG Model

In [ ]:
# this line loads the  dataset - we are using the extended connectivity fingerprints here, 
# there are several featurizers for you to try
# the splitter splits the dataset for you, refer to the MoleculeNet paper to see which one you need

featurizer = dc.feat.ConvMolFeaturizer()
tasks, datasets, transformers = dc.molnet.load_qm7(
    featurizer = featurizer,
    splitter='random')

# the datasets object is already split into the train, validation and test dataset 
train_dataset, valid_dataset, test_dataset = datasets
## N.B. Some molecules may not featurize and you'll get a warning this is OK

model = dc.models.DAGModel(
    n_tasks = len(test_dataset.tasks), # size of y, we have one output task here: finding toxicity
    #n_weave = 2,
    #fu_layer_sizes = [2000, 100],
    #n_atom_feat = 75,
    #n_pair_feat = 14,
    #n_hidden = 50,
    #n_graph_feat = 128,
    #conv_weight_init_stddevs = 0.03,
    weight_init_stddevs = 0.01,
    bias_init_consts = 0.0,
    weight_decay_penalty = 0.0,
    weight_decay_penalty_type = "l2",
    dropouts = 0.2,
    #n_classes = 1,
    batch_size = 100,
    
    mode = "regression"
    )

############################################
# Now we fit the training dataset!         #
############################################
model.fit(test_dataset, nb_epoch=2)

## GraphConvModel

In [ ]:
# this line loads the  dataset - we are using the extended connectivity fingerprints here, 
# there are several featurizers for you to try
# the splitter splits the dataset for you, refer to the MoleculeNet paper to see which one you need

featurizer = dc.feat.ConvMolFeaturizer()
tasks, datasets, transformers = dc.molnet.load_qm7(
    featurizer = featurizer,
    splitter='random')

# the datasets object is already split into the train, validation and test dataset 
train_dataset, valid_dataset, test_dataset = datasets
## N.B. Some molecules may not featurize and you'll get a warning this is OK

model = dc.models.GraphConvModel(
    n_tasks = len(test_dataset.tasks), # size of y, we have one output task here: finding toxicity
    #n_weave = 2,
    #fu_layer_sizes = [2000, 100],
    #n_atom_feat = 75,
    #n_pair_feat = 14,
    #n_hidden = 50,
    #n_graph_feat = 128,
    #conv_weight_init_stddevs = 0.03,
    weight_init_stddevs = 0.01,
    bias_init_consts = 0.0,
    weight_decay_penalty = 0.0,
    weight_decay_penalty_type = "l2",
    dropouts = 0.2,
    #n_classes = 1,
    batch_size = 100,
    
    mode = "regression"
    )

############################################
# Now we fit the training dataset!         #
############################################
model.fit(test_dataset, nb_epoch=2)

In [ ]:
model.predict(test_dataset)[:5]

In [ ]:
[ord(x) for x in "hello"]

In [ ]:
from deepchem.feat import SmilesToImage
from rdkit import Chem

max_len = 20
featurizer = SmilesToImage(img_size=250, max_len=max_len, res=0.5)
example_smiles = ['CCCCO', 'CCC']
example_mols = [Chem.MolFromSmiles(smile) for smile in example_smiles]
featurized_smiles = featurizer.featurize(example_mols)

tasks, dataset, transformers = dc.molnet.load_qm7(
    featurizer='smiles2img', 
    split='random', 
    img_spec='std')
train, valid, test = dataset

model = dc.models.ChemCeption(
    img_spec='std', 
    n_tasks=len(tasks), 
    mode='regression')
model.fit(train, nb_epoch=2)

# Training on the topological features

In [ ]:
%%time
output_metrics_tf=topol_regression_experiment(
    dataset=topol_dataset,
    transformers=transformers_tf,
    Splitter_Object=Splitter_Object_tf,
    tasks=task_list,
    metrics=metrics,
    metric_selector=selected_metric,
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score', 
                   'mae_score', 
                   'rmse']
    )
output_metrics_tf

In [ ]:
%%time
output_metrics_pca=topol_regression_experiment(
    dataset=pca_dataset,
    transformers=transformers_pca,
    Splitter_Object=Splitter_Object_pca,
    tasks=task_list,
    metrics=metrics,
    metric_selector=selected_metric,
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                'pearson_r2_score', 
                'mae_score', 
                'rmse']
    )
output_metrics_pca

In [ ]:
%%time
output_metrics_pca_no_transform=no_transform_topol_regression_experiment(
    dataset=pca_dataset,
    Splitter_Object=Splitter_Object_pca,
    tasks=task_list,
    metrics=metrics,
    metric_selector=selected_metric,
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                'pearson_r2_score', 
                'mae_score', 
                'rmse']
    )
output_metrics_pca_no_transform

## Dataset loaders for the other inputs if you want to play around with them

In [ ]:
tasks, datasets, transformers_ECFP = dc.molnet.load_qm7(
    shard_size=2000, featurizer="ECFP", splitter="stratified")
train_dataset_ECFP, valid_dataset_ECFP, test_dataset_ECFP = datasets
len(train_dataset_ECFP) + len(valid_dataset_ECFP) + len(test_dataset_ECFP)

In [ ]:
featurizer_CM_eig = dc.feat.CoulombMatrixEig(max_atoms=23)
tasks, datasets, transformers_CM_eig = dc.molnet.load_qm7(
    shard_size=2000, featurizer=featurizer_CM_eig, splitter="stratified")
train_dataset_CM_eig, valid_dataset_CM_eig, test_dataset_CM_eig = datasets
len(train_dataset_CM_eig) + len(valid_dataset_CM_eig) + len(test_dataset_CM_eig)

In [ ]:
featurizer_weave = dc.feat.WeaveFeaturizer()
tasks, datasets, transformers_weave = dc.molnet.load_qm7(
    shard_size=2000, featurizer=featurizer_weave, splitter="stratified")
train_dataset_weave, valid_dataset_weave, test_dataset_weave = datasets
len(train_dataset_weave) + len(valid_dataset_weave) + len(test_dataset_weave)

In [ ]:
#featurizer_rdkit = dc.feat.RDKitDescriptors()
#tasks, datasets, transformers_rdkit = dc.molnet.load_qm7(
#    shard_size=2000, featurizer=featurizer_rdkit, splitter="stratified")
#train_dataset_rdkit, valid_dataset_rdkit, test_dataset_rdkit = datasets
#len(train_dataset_rdkit) + len(valid_dataset_rdkit) + len(test_dataset_rdkit)

## Controls

In [ ]:
#featurizer = dc.feat.OneHotFeaturizer()#

#tasks, datasets, transformers = dc.molnet.load_qm7(
#            shard_size=2000, featurizer=featurizer, splitter="stratified")
        


## need to install stuff to do these
#featurizer = dc.feat.PubChemFingerprint()

#featurizer = dc.feat.Mol2VecFingerprint()

#featurizer = dc.feat.OneHotFeaturizer()

In [ ]:
def dataset_selector(setting):
    """makes data inside experiment function
    """
    print('!!!!Make this function new for each new dataset!!!!')
    if setting == 'ECFP':
        tasks, datasets, transformers = dc.molnet.load_qm7(
            shard_size=2000, featurizer="ECFP", splitter="stratified")
    elif setting == 'CM_eig':
        featurizer_CM_eig = dc.feat.CoulombMatrixEig(max_atoms=23)
        tasks, datasets, transformers = dc.molnet.load_qm7(
            shard_size=2000, featurizer=featurizer_CM_eig, splitter="stratified")
    elif setting == 'rdkit':
        featurizer_rdkit = dc.feat.RDKitDescriptors()
        tasks, datasets, transformers = dc.molnet.load_qm7(
            shard_size=2000, featurizer=featurizer_rdkit, splitter="stratified")
        
    elif setting == 'MACCS':
        featurizer = dc.feat.MACCSKeysFingerprint()

        tasks, datasets, transformers = dc.molnet.load_qm7(
            shard_size=2000, featurizer=featurizer, splitter="stratified")
    elif setting == '1HOT':
        featurizer = dc.feat.OneHotFeaturizer()

        tasks, datasets, transformers = dc.molnet.load_qm7(
            shard_size=2000, featurizer=featurizer, splitter="stratified")
    elif setting == 'CM':
        featurizer = dc.feat.CoulombMatrix(max_atoms=23)
        tasks, datasets, transformers = dc.molnet.load_qm7(
            shard_size=2000, featurizer=featurizer, splitter="stratified")

    else:
        print('meh no data imported')
    return tasks, datasets, transformers

dataset_selector('CM')
#featurizer = dc.feat.CoulombMatrix(max_atoms=23)
#tasks, datasets, transformers = dc.molnet.load_qm7(
#    shard_size=2000, featurizer=featurizer, splitter="stratified")
#train_dataset, valid_dataset, test_dataset = datasets

#model=dc.models.DTNNModel(n_tasks = len(tasks), nb_epochs=3)
#model.fit(train_dataset, nb_epoch=3)

## ECFP

In [ ]:

def deepchem_regression_experiment(
    metrics,
    metric_selector,
    setting='ECFP',
    num_repeats=5,
    num_epochs=250,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse']):
    """runs repeated training with topol input features
    uses default multitask regressor class
    does splitting and transforming
    returns metrics
    
    dataset: overall original dataset, before splitting
    transformers: deepchem transformer
    Splitter_object: deepchem splitter
    num_epochs: num epochs if not early stopping
    metric_selector: whihc one to train wiht
    split fraction: train, val, test split as decimal
    patience: for early stopping
    metric_labels: add rmse here if you want it, do not add to metrics
    metrics: list of metrics"""

## in function

    out=[]

    frac_train=split_fraction[0]
    frac_valid=split_fraction[1]
    frac_test=split_fraction[2]
    train_scores, validate_scores, test_scores = [],[],[]
    print(f'Metric selected is {metric_labels[metric_selector]}')
    for i in range(num_repeats):
        ################### this bit is different ##############
        print(f'Using dataset selector setting {setting}')
        tasks, datasets, transformers=dataset_selector(setting)
        train_dataset, valid_dataset, test_dataset = datasets
        task_list = tasks
        ########################################################
        # actual model here
        model = dc.models.MultitaskRegressor(
            n_tasks=len(tasks),
            n_features=len(train_dataset.X[3]),
            #layer_sizes=[1000,1000,500,20],
            #dropouts=0.2,
            #learning_rate=0.001,
            residual=True)
        callback = dc.models.ValidationCallback(
            valid_dataset, 
            patience, 
            metrics[metric_selector])
        # fit da model
        model.fit(train_dataset, nb_epoch=num_epochs, callbacks=callback)

        # little function to calc metrics on this data
        out.append(get_them_metrics(
            model,
            datasets,
            metrics,
            metric_labels,
            transformers))
    
    pd_out =pd.DataFrame(out, columns=['tr_mse', 'tr_r2', 'tr_mae','tr_rmse',
                              'val_mse', 'val_r2', 'val_mae','val_rmse',
                              'te_mse','te_r2', 'te_mae', 'te_rmse'])
    return pd_out




In [ ]:

def deepchem_CM_regression_experiment(
    metrics,
    metric_selector,
    setting='ECFP',
    num_repeats=5,
    num_epochs=250,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse']):
    """runs repeated training with topol input features
    uses default multitask regressor class
    does splitting and transforming
    returns metrics
    
    dataset: overall original dataset, before splitting
    transformers: deepchem transformer
    Splitter_object: deepchem splitter
    num_epochs: num epochs if not early stopping
    metric_selector: whihc one to train wiht
    split fraction: train, val, test split as decimal
    patience: for early stopping
    metric_labels: add rmse here if you want it, do not add to metrics
    metrics: list of metrics"""

## in function

    out=[]

    frac_train=split_fraction[0]
    frac_valid=split_fraction[1]
    frac_test=split_fraction[2]
    train_scores, validate_scores, test_scores = [],[],[]
    print(f'Metric selected is {metric_labels[metric_selector]}')
    for i in range(num_repeats):
        ################### this bit is different ##############
        print(f'Using dataset selector setting {setting}')
        tasks, datasets, transformers=dataset_selector(setting)
        train_dataset, valid_dataset, test_dataset = datasets
        task_list = tasks
        ########################################################
        # actual model here
        model=dc.models.DTNNModel(n_tasks = len(tasks), nb_epochs=3)
        callback = dc.models.ValidationCallback(
            valid_dataset, 
            patience, 
            metrics[metric_selector])
        # fit da model
        model.fit(train_dataset, nb_epoch=num_epochs, callbacks=callback)

        # little function to calc metrics on this data
        out.append(get_them_metrics(
            model,
            datasets,
            metrics,
            metric_labels,
            transformers))
    
    pd_out =pd.DataFrame(out, columns=['tr_mse', 'tr_r2', 'tr_mae', 'tr_rmse',
                              'val_mse', 'val_r2', 'val_mae','val_rmse',
                              'te_mse','te_r2', 'te_mae',  'te_rmse'])
    return pd_out




In [ ]:
%%time
output_metrics_cm=deepchem_CM_regression_experiment(
    metrics,
    metric_selector=2,
    setting='CM',
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])
output_metrics_cm.head(3)

# Control experiments:
regression first

In [ ]:
%%time
output_metrics_ecfp = deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    setting='ECFP',
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])
output_metrics_ecfp

### Coulomb matrix eigenvalues

In [ ]:
%%time
output_metrics_cm_eig = deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    setting='CM_eig',
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])
output_metrics_cm_eig

### rdkit features

In [ ]:
%%time
output_metrics_rdkit = deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    setting='rdkit',
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])
output_metrics_rdkit

## MACCS features

In [ ]:
%%time
output_metrics_maccs = deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    setting='MACCS',
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])
output_metrics_maccs

In [ ]:
#Best=1.92

data=[output_metrics_ecfp['tr_mae'],
     output_metrics_ecfp['te_mae'],
     output_metrics_maccs['tr_mae'],
     output_metrics_maccs['te_mae'],
     output_metrics_rdkit['tr_mae'],
     output_metrics_rdkit['te_mae'],
     output_metrics_cm['tr_mae'],
     output_metrics_cm['te_mae'],
     output_metrics_cm_eig['tr_mae'],
     output_metrics_cm_eig['te_mae'],
     output_metrics_tf['tr_mae'],
     output_metrics_tf['te_mae'],
     output_metrics_pca_no_transform['tr_mae'],
     output_metrics_pca_no_transform['te_mae']]
      
means=[np.mean(x) for x in data]
stds=[np.std(x) for x in data]


x_positions = [x+1 for x in range(len(means))]
plt.figure(figsize=(16, 9))
plt.rcParams.update({'font.size': 22})

plt.bar(x_positions, means, 
        color=["#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#88fff4","#552200",
               "#88fff4","#552200",
               "#5f77f4","#f62788",
               "#5f77f4","#f62788"], 
        edgecolor='k',
        linewidth='3',
        yerr=stds, 
        error_kw=dict(lw=5, capsize=15, capthick=3))
for i in range(len(means)):
    x=data[i]; plt.plot(np.ones(len(x))*(i+1),x,'o',color="#444444")

plt.plot([0.5,14.5],[153, 153],'k',linewidth=4,linestyle='-.')
axes=plt.gca()
#axes.set_ylim([0.5,4.5])
#axes.set_xlim([0.45,4.55])
plt.xticks([1.5,3.5,5.5,7.5,9.5,11.5,13.5],['ECFP',"MACCS","rdkit","CM","CM_eig","TDAF","PCA-TDAF"])
plt.ylabel('MAE kcal/mol')
#plt.savefig("dc_NN_MR_rmse.png")

In [ ]:
list_of_data_frames=[output_metrics_ecfp,
     output_metrics_maccs,
     output_metrics_rdkit,
     output_metrics_cm,
     output_metrics_cm_eig,
     output_metrics_tf,
     output_metrics_pca_no_transform]

list_of_data_frames_names=["output_metrics_ecfp",
     "output_metrics_maccs",
     "output_metrics_rdkit",
     "output_metrics_cm",
     "output_metrics_cm_eig",
     "output_metrics_tf",
     "output_metrics_pca_no_transform"]

In [ ]:
list_of_data_frames_names[4] + '.csv'

In [ ]:
for i in range(len(list_of_data_frames)):
    list_of_data_frames[i].to_csv(os.path.join(
        results_dir, 
        list_of_data_frames_names[i] + '.csv'))


# stuff that doesn't work yet

# graph convolution

In [ ]:
import numpy
numpy.version.version

In [ ]:
featurizer = dc.feat.ConvMolFeaturizer()
tasks, datasets, transformers_gc = dc.molnet.load_qm7(
    shard_size=2000, featurizer=featurizer, splitter="stratified")
train_dataset, valid_dataset, test_dataset = datasets
n_tasks = len(tasks)
model = dc.models.GraphConvModel(n_tasks, mode='regression')
model.fit(train_dataset, nb_epoch=50)

In [ ]:
output_metrics_1hot = deepchem_regression_experiment(
    metrics,
    metric_selector=selected_metric,
    setting='1HOT',
    num_repeats=num_repeats,
    num_epochs=num_epochs,
    split_fraction=[0.8,0.1,0.1],
    patience=15,
    metric_labels=['mean_squared_error',
                   'pearson_r2_score',
                   'mae_score', 
                   'rmse'])
output_metrics_1hot

In [ ]:
model = dc.models.GraphConvModel(n_tasks = len(tasks),mode='regression')
model.fit( train_dataset, nb_epoch=3)

In [ ]:
n_tasks = len(tasks)
model = dc.models.GraphConvModel(n_tasks, mode='regression')
model.fit(train_dataset, nb_epoch=50)

In [ ]:
featurizer = dc.feat.CoulombMatrix(max_atoms=23)
tasks, datasets, transformers = dc.molnet.load_qm7(
    shard_size=2000, featurizer=featurizer, splitter="stratified")
train_dataset, valid_dataset, test_dataset = datasets

model=dc.models.DTNNModel(n_tasks = len(tasks), nb_epochs=3)
model.fit(train_dataset, nb_epoch=3)

In [ ]:
featurizer_weave = dc.feat.WeaveFeaturizer()
tasks, datasets, transformers_weave = dc.molnet.load_qm7(
    shard_size=2000, featurizer=featurizer_weave, splitter="stratified")
train_dataset_weave, valid_dataset_weave, test_dataset_weave = datasets
len(train_dataset_weave) + len(valid_dataset_weave) + len(test_dataset_weave)

In [ ]:
featurizer = dc.feat.MACCSKeysFingerprint()

featurizer = dc.feat.CircularFingerprint(size=2048, radius=4)

featurizer = dc.feat.PubChemFingerprint()

featurizer = dc.feat.Mol2VecFingerprint()

featurizer = dc.feat.OneHotFeaturizer()

In [ ]:
import deepchem as dc
featurizers = dc.feat.CoulombMatrix(max_atoms=23)
input_file = 'deepchem/feat/tests/data/water.sdf' # really backed by water.sdf.csv
tasks = ["atomization_energy"]
loader = dc.data.SDFLoader(tasks, featurizer=featurizers)
dataset = loader.create_dataset(input_file)

In [ ]:
featurizer = dc.feat.BPSymmetryFunctionInput(max_atoms=10)

In [ ]:
smiles = ['CC(=O)OC1=CC=CC=C1C(=O)O']
featurizer = dc.feat.SmilesToImage(img_size=80, img_spec='std')
images = featurizer.featurize(smiles)
type (images[0])

images[0].shape # (img_size, img_size, 1)

In [ ]:
!pip install transformers
from deepchem.feat.smiles_tokenizer import SmilesTokenizer
current_dir = os.path.dirname(os.path.realpath(__file__))
vocab_path = os.path.join(current_dir, 'tests/data', 'vocab.txt')
tokenizer = SmilesTokenizer(vocab_path)
print(tokenizer.encode("CC(=O)OC1=CC=CC=C1C(=O)O"))

In [ ]:
smiles = ["C", "CCC"]
featurizer=dc.feat.ConvMolFeaturizer(per_atom_fragmentation=False)
f = featurizer.featurize(smiles)
# Using ConvMolFeaturizer to create featurized fragments derived from molecules of interest.
# This is used only in the context of performing interpretation of models using atomic
# contributions (atom-based model interpretation)
smiles = ["C", "CCC"]
featurizer=dc.feat.ConvMolFeaturizer(per_atom_fragmentation=True)
f = featurizer.featurize(smiles)
len(f)

In [ ]:
featurizer = dc.feat.OneHotFeaturizer()
smiles = ['CCC']
encodings = featurizer.featurize(smiles)
type(encodings[0])

encodings[0].shape
(100, 35)
featurizer.untransform(encodings[0])
'CCC'

In [ ]:
encodings

In [ ]:
metric1 = dc.metrics.Metric(dc.metrics.mae_score)
metric2 = dc.metrics.Metric(dc.metrics.pearson_r2_score)
metrics = [metric1, metric2]
    
train_scores_weave, validate_scores_weave, test_scores_weave = [],[],[]
for i in range(10):
    model = dc.models.DTNNModel(
        n_tasks=len(tasks))
    callback = dc.models.ValidationCallback(valid_dataset_weave, 5, metric1)
    model.fit(train_dataset_weave, nb_epoch=25, callbacks=callback)

    # line below returns a dictionary
    #train_scores_ecfp.append(model.evaluate(train_dataset_ECFP, 
                           #            metric1)['mae_score'])
    #validate_scores_ecfp.append(model.evaluate(valid_dataset_ECFP, 
                            #              metric1)['mae_score'])
    #test_scores_ecfp.append(model.evaluate(test_dataset_ECFP, 
                             #         metric1)['mae_score'])
    
        # line below returns a dictionary
    train_scores_weave.append(
        model.evaluate(
            train_dataset_weave, 
            metric1,
            transformers = transformers_weave)['mae_score'])
    validate_scores_weave.append(
        model.evaluate(
            valid_dataset_weave, 
            metric1,transformers = transformers_weave)['mae_score']
            )
    test_scores_weave.append(
        model.evaluate(
            test_dataset_weave, 
            metric1,transformers = transformers_weave)['mae_score']
            )

## Weave

In [ ]:
test_dataset_weave

In [ ]:
import scipy
scipy.stats.ttest_ind(test_scores_CM_eig, test_scores_pca, axis=0, equal_var=True, nan_policy='propagate', alternative='two-sided')

In [ ]:
scipy.stats.ttest_ind(test_scores_CM_eig, test_scores_tf, axis=0, equal_var=True, nan_policy='propagate', alternative='two-sided')

In [ ]:
scipy.stats.ttest_ind(test_scores_rdkit, test_scores_tf, axis=0, equal_var=True, nan_policy='propagate', alternative='two-sided')

In [ ]:
scipy.stats.ttest_ind(test_scores_rdkit, test_scores_pca, axis=0, equal_var=True, nan_policy='propagate', alternative='two-sided')

In [ ]:
means

In [ ]:
output_metrics_tf['tr_mae']

In [ ]:
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer='GraphConv')
train_dataset, valid_dataset, test_dataset = datasets
n_tasks = len(tasks)
model = dc.models.GraphConvModel(n_tasks, mode='classification')
model.fit(train_dataset, nb_epoch=50)

## plots the graph

In [ ]:
#Best=1.92

data=[output_metrics_ecfp['tr_mae'],
     output_metrics_ecfp['te_mae'],
     output_metrics_rdkit['tr_mae'],
     output_metrics_rdkit['te_mae'],
     output_metrics_maccs['tr_mae'],
     output_metrics_maccs['te_mae'],
     output_metrics_cm['tr_mae'],
     output_metrics_cm['te_mae'],
     output_metrics_cm_eig['tr_mae'],
     output_metrics_cm_eig['te_mae'],
     output_metrics_tf['tr_mae'],
     output_metrics_tf['te_mae'],
     output_metrics_pca_no_transform['tr_mae'],
     output_metrics_pca_no_transform['te_mae']]
      
means=[np.mean(x) for x in data]
stds=[np.std(x) for x in data]


x_positions = [x+1 for x in range(len(means))]
plt.figure(figsize=(16, 9))
plt.rcParams.update({'font.size': 22})

plt.bar(x_positions, means, 
        color=["#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#1f77f4","#f62728",
               "#88fff4","#552200",
               "#5f77f4","#f62788",
               "#5f77f4","#f62788"], 
        edgecolor='k',
        linewidth='3',
        yerr=stds, 
        error_kw=dict(lw=5, capsize=15, capthick=3))
for i in range(len(means)):
    x=data[i]; plt.plot(np.ones(len(x))*(i+1),x,'o',color="#444444")

plt.plot([0.5,15.5],[153, 153],'k',linewidth=4,linestyle='-.')
axes=plt.gca()
#axes.set_ylim([0.5,4.5])
#axes.set_xlim([0.45,4.55])
plt.xticks([1.5,3.5,5.5,7.5,9.5,11.5,13.5],['ECFP',"rdkit","MACCS","CM","CM_eig","TDAF","PCA-TDAF"])
plt.ylabel('MAE kcal/mol')
#plt.savefig("dc_NN_MR_rmse.png")

## Broken shit

In [ ]:
import string
def build_char_dict():
    cti = {}
    for i, x in enumerate(string.printable):
        cti[x] = i

    cti['<unk>'] = max(cti.values()) + 1
    cti['<pad>'] = max(cti.values()) + 1
    return cti
char_to_idx=build_char_dict()

print(char_to_idx)
featurizer = dc.feat.SmilesToSeq(char_to_idx=char_to_idx)
print(str(featurizer))
print(featurizer.to_seq('CCOc1ccc2nc(S(N)(=O)=O)sc2c1'))

tasks, datasets, transformers = dc.molnet.load_qm7(
    featurizer=featurizer,
    splitter='random')

# the datasets object is already split into the train, validation and test dataset
train_dataset, valid_dataset, test_dataset = datasets
## N.B. Some molecules may not featurize and you'll get a warning this is OK

model = dc.models.Smiles2Vec(
    char_to_idx=char_to_idx,
    n_tasks=len(test_dataset.tasks), # size of y, we have one output task here: finding toxicity
    mode="regression",
    embedding_dim=len(char_to_idx)
    )

model.fit(test_dataset, nb_epoch=2)

model.predict(dataset=test_dataset,
              transformers=transformers)[:5]

## Controls

In [ ]:
params_dict = {
    'n_tasks': [len(tasks)],
    'n_features': [len(my_dataset.X[0])],
    'layer_sizes': [[500], [1000], [1000, 1000]],
    'dropouts': [0.2, 0.5],
    'learning_rate': [0.001, 0.0001]
}
optimizer = dc.hyper.GridHyperparamOpt(dc.models.MultitaskRegressor)
#metric = dc.metrics.Metric(dc.metrics.rms_score)
# We want to know the RMS, averaged across tasks
metric = dc.metrics.Metric(dc.metrics.rms_score, np.mean)
best_model, best_hyperparams, all_results = optimizer.hyperparam_search(
    params_dict, 
    train_dataset_ECFP, 
    valid_dataset_ECFP, 
    metric, 
    transformers)

In [ ]:
metric1 = dc.metrics.Metric(dc.metrics.mae_score)
metric2 = dc.metrics.Metric(dc.metrics.pearson_r2_score)
metrics = [metric1, metric2]
    
train_scores_ECFP, validate_scores_ECFP, test_scores_ECFP = [],[],[]
for i in range(1):
    model = dc.models.MultitaskRegressor(
        n_tasks=len(tasks),
        n_features=1024,
        layer_sizes=[1000,1000],
        dropouts=0.4,
        learning_rate=0.01)
    callback = dc.models.ValidationCallback(valid_dataset_ECFP, 5, metric1)
    model.fit(train_dataset_ECFP, nb_epoch=250, callbacks=callback)

    # line below returns a dictionary
    train_scores_ECFP.append(model.evaluate(train_dataset_ECFP, 
                                       metric1)['mae_score'])
    validate_scores_ECFP.append(model.evaluate(valid_dataset_ECFP,
                                          metric1)['mae_score'])
    test_scores_ECFP.append(model.evaluate(test_dataset_ECFP,
                                      metric1)['mae_score'])

In [ ]:
test_scores_ECFP

In [ ]:

transformers[1].untransform(test_scores_ECFP[0])

In [ ]:
transformers[1].untransform(best_krr.model.score(test_dataset.X, test_dataset.y))

In [ ]:
# This loads the data without doing any featurization
# currently does a normalisation transformation, this can be removed
# the splitter does not shuffle the data
tasks, datasets, transformers = dc.molnet.load_qm7(
    shard_size=2000, 
    featurizer=h.My_Dummy_Featurizer(None), 
    splitter="index")
sdf_train_dataset, sdf_valid_dataset, sdf_test_dataset = datasets

In [ ]:

sdf_data_dir=r'F:\Nextcloud\science\Datasets'


In [ ]:
tasks = ["atomization_energy"]
sdf_filename='gdb7.sdf'
#dataset_file = "../../datasets/gdb1k.sdf"
smiles_field = "smiles"
mol_field = "mol"

In [ ]:
featurizer_CM = dc.feat.CoulombMatrixEig(23, remove_hydrogens=False)
loader = dc.data.SDFLoader(
      tasks=["u0_atom"],
      featurizer=featurizer_CM)
dataset = loader.create_dataset(os.path.join(sdf_data_dir, sdf_filename))

In [ ]:

random_splitter = dc.splits.RandomSplitter()
train_dataset, valid_dataset, test_dataset = random_splitter.train_valid_test_split(dataset)

In [ ]:
transformers = [
    dc.trans.NormalizationTransformer(transform_X=True, dataset=train_dataset),
    dc.trans.NormalizationTransformer(transform_y=True, dataset=train_dataset)]

for dataset in [train_dataset, valid_dataset, test_dataset]:
  for transformer in transformers:
      dataset = transformer.transform(dataset)

In [ ]:
def rf_model_builder(model_dir, **model_params):
  sklearn_model = RandomForestRegressor(**model_params)
  return dc.models.SklearnModel(sklearn_model, model_dir)
params_dict = {
    "n_estimators": [10, 100],
    "max_features": ["auto", "sqrt", "log2", None],
}

metric = dc.metrics.Metric(dc.metrics.mean_absolute_error)
optimizer = dc.hyper.GridHyperparamOpt(rf_model_builder)
best_rf, best_rf_hyperparams, all_rf_results = optimizer.hyperparam_search(
    params_dict, train_dataset, valid_dataset, output_transformers=transformers,
    metric=metric, use_max=False)
for key, value in all_rf_results.items():
    print(f'{key}: {value}')
print('Best hyperparams:', best_rf_hyperparams)

In [ ]:

def krr_model_builder(model_dir, **model_params):
  sklearn_model = KernelRidge(**model_params)
  return dc.models.SklearnModel(sklearn_model, model_dir)

params_dict = {
    "kernel": ["laplacian"],
    "alpha": [0.0001],
    "gamma": [0.0001]
}

metric = dc.metrics.Metric(dc.metrics.mean_absolute_error)
optimizer = dc.hyper.GridHyperparamOpt(krr_model_builder)
best_krr, best_krr_hyperparams, all_krr_results = optimizer.hyperparam_search(
    params_dict, train_dataset, valid_dataset, output_transformers=transformers,
    metric=metric, use_max=False)
for key, value in all_krr_results.items():
    print(f'{key}: {value}')
print('Best hyperparams:', best_krr_hyperparams)

In [ ]:
best_krr.model.score(test_dataset.X, test_dataset.y)

In [ ]:
untransformed_train_y = transformers[0].untransform(orig_test_dataset.y)
my_untransformed_dataset = dc.data.NumpyDataset(X=X_data, y=untransformed_train_y[0:69])

In [ ]:
transformers[1].untransform(best_krr.model.score(test_dataset.X, test_dataset.y))

In [ ]:
fh.close()